<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-01/NB01_what_is_ai.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 01: What Is Artificial Intelligence? Definitions, Aims and Measures

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-01/lab.html>.

## Overview

Before asking whether machines can think, it helps to ask what artificial intelligence is supposed to be. This week compares definitions of AI, separates strong from weak AI and examines formal attempts to measure intelligence, which make the hidden assumptions of informal definitions explicit [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to distinguish the four approaches in the classification of Russell and Norvig, to explain the difference between strong and weak AI and between both and general AI, to compute a simplicity-weighted intelligence score in the spirit of Legg and Hutter, and to discuss Chollet's view of intelligence as skill-acquisition efficiency.

## A field defined by an ambition

The proposal for the 1955 Dartmouth summer project, which gave the field its name, rested on the conjecture that every aspect of learning and intelligence can in principle be described precisely enough for a machine to simulate it [4]. The conjecture is philosophical as much as technical: It assumes that intelligence is a matter of describable processes rather than of a particular biological substrate.

Russell and Norvig classify definitions of AI along two dimensions [1]. The first asks whether a definition concerns thought or behaviour. The second asks whether success is measured against human performance or against an ideal of rationality. The result is four approaches. Thinking humanly is the programme of cognitive modelling, which compares programs with human thought processes. Thinking rationally follows the laws of thought tradition, in which intelligence is correct inference. Acting humanly is the approach of the Turing test, which Week 2 examines. Acting rationally, the approach that Russell and Norvig adopt, defines an intelligent agent as one that acts to achieve the best expected outcome. Figure 1.1 summarises the classification.

![Figure 1.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-01/figures/w01_fig1.png)

*Figure 1.1. The classification of definitions of AI by Russell and Norvig along two dimensions: thought versus behaviour and human performance versus rationality [1].*

## Strong and weak AI

Searle introduced a distinction that frames much of this course [2]. According to weak AI, the computer is a powerful tool for studying the mind, for example by testing hypotheses about cognition with simulations. According to strong AI, an appropriately programmed computer does not merely simulate a mind: It literally has cognitive states and understands. Searle accepted weak AI and rejected strong AI, and his argument is the topic of Week 4.

The distinction is often confused with a different one. Narrow AI performs specific tasks, while general AI would perform well across a wide range of tasks. This is a distinction of capability, not of metaphysics. A system could be general without having a mind in Searle's sense, and a narrow system could in principle be a candidate for strong AI. Keeping the two axes apart prevents a common error, namely inferring conclusions about minds from improvements in performance.

The philosophy of AI therefore asks several kinds of question. Conceptual questions ask what thinking, understanding and intelligence are. Metaphysical questions ask whether a machine could have a mind. Epistemological questions ask how anyone could know. Methodological questions ask what AI research shows about minds in general.

## Measuring intelligence

Informal definitions leave room for disagreement, so some researchers have tried to formalise intelligence. Legg and Hutter collected many informal definitions and proposed universal intelligence: the expected performance of an agent across all computable environments, where each environment is weighted by two to the power of minus its Kolmogorov complexity [3]. Simple environments therefore count more than complex ones. The measure cannot be computed exactly, because Kolmogorov complexity is not computable, but it can be approximated, and its structure exposes a choice that informal definitions hide: which environments matter, and how much.

Chollet argued that performance on any fixed set of tasks measures skill, which can be bought with prior knowledge or unlimited training data [5]. He proposed to define intelligence as skill-acquisition efficiency: how efficiently a system turns its priors and experience into skill on tasks that require generalisation. His benchmark of abstract reasoning puzzles, ARC, was designed on this principle. The lab of this week shows how the ranking of four agents changes when the weighting of environments changes, which makes the dependence of any intelligence score on its assumptions visible.

## Why definitions matter

Definitions are not neutral. A benchmark score that improves on narrow tasks may be reported as progress towards general intelligence, and a behavioural definition may be read as settling questions about understanding. Each later week of the course tests one of these inferences. Weeks 2 to 4 examine behaviour, computation and understanding. Weeks 5 to 8 turn to meaning, cognitive architecture, embodiment and common sense. Week 9 examines arguments that machines face principled limits. Weeks 10 and 11 address consciousness. Weeks 12 to 14 apply the tools of the course to understanding, creativity and intentionality in present systems.

> **Pause and reflect.** Which of the four approaches best describes the AI systems in your own research? Would your answer change if the systems became much more capable?

# Hands-on lab

The notebook builds six sequence-prediction environments from short programs, approximates their description length by compression, runs four prediction agents in each and computes simplicity-weighted scores, following the structure of the universal intelligence measure [3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Environments generated by programs

Each environment is a binary sequence. Its description length is approximated by the size in bits of the compressed sequence, a rough stand-in for Kolmogorov complexity, which cannot be computed exactly.

In [ ]:
import zlib
import pandas as pd

T = 600
def is_prime(n):
    return n > 1 and all(n % d for d in range(2, int(n ** 0.5) + 1))

lcg = [0]
def pseudo_random(t):
    lcg[0] = (1103515245 * lcg[0] + 12345) % 2 ** 31
    return (lcg[0] >> 16) & 1

ENVS = {
    "constant": [1] * T,
    "alternating": [t % 2 for t in range(T)],
    "period 3": [int(t % 3 != 2) for t in range(T)],
    "period 7": [[1, 0, 0, 1, 1, 0, 1][t % 7] for t in range(T)],
    "primes": [int(is_prime(t)) for t in range(T)],
    "pseudo-random": [pseudo_random(t) for t in range(T)],
}
K = {name: 8 * len(zlib.compress(bytes(seq), 9)) for name, seq in ENVS.items()}
pd.Series(K, name="approximate description length (bits)")

## 2. Four agents

In [ ]:
from collections import Counter, defaultdict

def constant_agent(history):
    return 1

def majority_agent(history):
    return int(sum(history) * 2 >= len(history)) if history else 1

def ngram_agent(history, k=3):
    if len(history) <= k:
        return majority_agent(history)
    counts = defaultdict(Counter)
    for i in range(k, len(history)):
        counts[tuple(history[i - k:i])][history[i]] += 1
    c = counts.get(tuple(history[-k:]))
    return c.most_common(1)[0][0] if c else majority_agent(history)

def period_agent(history, max_period=8, window=32):
    recent = history[-window:]
    for p in range(1, max_period + 1):
        if len(recent) > p and all(recent[i] == recent[i - p] for i in range(p, len(recent))):
            return recent[-p]
    return majority_agent(history)

AGENTS = {"constant": constant_agent, "majority": majority_agent, "3-gram": ngram_agent, "period finder": period_agent}

def accuracy(agent, seq):
    return np.mean([agent(seq[:t]) == seq[t] for t in range(1, len(seq))])

ACC = pd.DataFrame({a: {e: accuracy(f, s) for e, s in ENVS.items()} for a, f in AGENTS.items()}).round(3)
ACC

## 3. Simplicity-weighted scores

With a prior strength alpha, environment e receives the weight 2^(-alpha K(e)), normalised to sum to one. With alpha = 0, every environment counts equally.

### Your turn, exercise 1

Write `weighted_scores(alpha)` that returns a pandas Series with the weighted score of every agent.

In [ ]:
def weighted_scores(alpha):
    # Your code here
    return None

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _weighted_reference(alpha):
    k = pd.Series(K)
    w = 2.0 ** (-alpha * (k - k.min()))
    w = w / w.sum()
    return ACC.mul(w, axis=0).sum()

In [ ]:
attempt = weighted_scores(0.01)
check("Exercise 1", None if attempt is None else attempt.values, _weighted_reference(0.01).values)
alphas = np.linspace(0, 0.03, 31)
curves = pd.DataFrame({a: _weighted_reference(a) for a in alphas}).T
curves.plot(figsize=(7, 3.6)); plt.xlabel("alpha (strength of the simplicity prior)"); plt.ylabel("weighted score")
plt.title("The ranking depends on how environments are weighted"); plt.show()

Subtracting the smallest description length before exponentiating does not change the normalised weights; it only avoids numerical underflow.

### Your turn, exercise 2

Return the ranking of the four agents, from best to worst, for alpha = 0 as a list of names. Compare it with the ranking for alpha = 0.03 printed below, find the pair of agents that swap places and write three sentences on what the swap implies for claims that one system is more intelligent than another.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _ranking_reference(alpha):
    return list(_weighted_reference(alpha).sort_values(ascending=False).index)

In [ ]:
ranking_at_zero = None  # a list of agent names, best first
check("Exercise 2", ranking_at_zero, _ranking_reference(0.0))
print("ranking for alpha = 0.03:", _ranking_reference(0.03))

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-01/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Write your own definition of intelligence in one sentence. Which of the four approaches does it belong to, and what would count as evidence against it?
2. Is it possible for a system to be general but not strong in Searle's sense? Give an example or an argument.
3. Which environments would you include in an intelligence measure for systems in your field, and how would you weight them?

## Weekly task and submission

Write a short essay of about 500 words that compares two definitions of intelligence from this week, identifies an assumption that each makes explicit or hides, and applies both to one AI system of your choice. Cite at least three works in square brackets and attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**What do benchmarks say intelligence is?.** Analyse how three current AI benchmarks operationalise intelligence and compare them with the formal measures of this week [3, 5]. State which approach in the classification of Russell and Norvig each benchmark follows.

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

[2] Searle, J. R. (1980). Minds, brains, and programs. *Behavioral and Brain Sciences*, 3(3), 417-424. <https://doi.org/10.1017/S0140525X00005756>

[3] Legg, S., & Hutter, M. (2007). Universal intelligence: A definition of machine intelligence. *Minds and Machines*, 17(4), 391-444. <https://doi.org/10.1007/s11023-007-9079-x>

[4] McCarthy, J., Minsky, M. L., Rochester, N., & Shannon, C. E. (2006). A proposal for the Dartmouth summer research project on artificial intelligence, August 31, 1955. *AI Magazine*, 27(4), 12-14. <https://doi.org/10.1609/aimag.v27i4.1904>

[5] Chollet, F. (2019). On the measure of intelligence. arXiv preprint arXiv:1911.01547. <https://arxiv.org/abs/1911.01547>